# Job Scraper — Jupyter Notebook

**Goal:** Build a job-scraper that collects job listings (example: Indeed) and stores them in SQLite.
**Tech stack:** Python, requests + BeautifulSoup (primary), Selenium fallback, pandas, sqlite3

This notebook contains:
- configuration & dependencies
- helper functions: session, headers rotation, rate limiting
- scraping with pagination + parsing
- storing and deduplication in SQLite
- optional Selenium driver for JS-rendered pages
- example run and troubleshooting notes

**Notes:** This is an educational/comprehensive template — adapt selectors and crawling etiquette for the target site and your usage.

In [ ]:
# Install instructions (run in a cell if running in a fresh environment)
# Uncomment and run these if packages are missing.
# !pip install requests beautifulsoup4 pandas sqlalchemy lxml selenium webdriver-manager tqdm

## 1) Configuration
- Choose target: `indeed.com` (example). Many job sites have anti-scraping measures — respect robots.txt and terms of service. For production, use official APIs when available.

In [9]:
import time
import random
import sqlite3
import logging
from typing import List, Dict, Optional, Tuple
from urllib.parse import urljoin, urlencode

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm


# Optional imports for Selenium fallback
try:
	from selenium import webdriver
	from selenium.webdriver.chrome.options import Options
	from webdriver_manager.chrome import ChromeDriverManager
	SELENIUM_AVAILABLE = True
except Exception:
	SELENIUM_AVAILABLE = False

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')    

### Basic config

In [10]:
CONFIG = {
'base_url': 'https://www.indeed.com/jobs', # example: indeed search endpoint
'query': 'data analyst',
'location': 'Egypt',
'results_per_page': 10, # site-specific
'max_pages': 5,
'delay_min': 1.0,
'delay_max': 3.0,
'sqlite_db': 'jobs.db',
}


# A small pool of common user-agents for polite rotation
USER_AGENTS = [
'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.2 Safari/605.1.15',
'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
]

## 2) Database: SQLite helper functions
- Create table
- Insert with deduplication (unique constraint)

In [11]:
CREATE_TABLE_SQL = '''
CREATE TABLE IF NOT EXISTS jobs (
id INTEGER PRIMARY KEY AUTOINCREMENT,
job_id TEXT, -- site-specific id if available
title TEXT,
company TEXT,
location TEXT,
summary TEXT,
post_date TEXT,
url TEXT,
raw_html TEXT,
scraped_at TEXT DEFAULT (datetime('now')),
UNIQUE(job_id, title, company) ON CONFLICT IGNORE
);
'''

In [13]:
def init_db(db_path: str = CONFIG['sqlite_db']):
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()
	cur.execute(CREATE_TABLE_SQL)
	conn.commit()
	conn.close()
	logging.info(f"Initialized DB at {db_path}")

In [14]:
def save_jobs_to_db(jobs: List[Dict], db_path: str = CONFIG['sqlite_db']):
	if not jobs:
		return 0
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()
	inserted = 0
	for j in jobs:
		try:
			cur.execute('''INSERT OR IGNORE INTO jobs
			(job_id, title, company, location, summary, post_date, url, raw_html)
			VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
			(j.get('job_id'), j.get('title'), j.get('company'), j.get('location'),
			j.get('summary'), j.get('post_date'), j.get('url'), j.get('raw_html')))
			if cur.rowcount:
				inserted += 1
		except Exception as e:
			logging.exception('Error inserting job: %s', e)
	conn.commit()
	conn.close()
	logging.info(f"Inserted {inserted} new jobs to {db_path}")
	return inserted

## 3) HTTP helpers: session, headers, rate-limiting

In [15]:
def random_headers():
	return {
		'User-Agent': random.choice(USER_AGENTS),
		'Accept-Language': 'en-US,en;q=0.9',
		'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
	}


# A requests session that can be reused
def get_session():
	s = requests.Session()
	s.headers.update(random_headers())
	return s

## 4) Parsers: adapt these functions to the target site's HTML structure


We'll provide an example parser for Indeed-like markup. You must inspect the site's current HTML (use browser DevTools) and adapt selectors accordingly.

In [16]:
def parse_indeed_search_page(html: str, base_url: str = 'https://www.indeed.com') -> List[Dict]:
	soup = BeautifulSoup(html, 'lxml')
	results = []
	# Indeed uses divs with class 'jobsearch-SerpJobCard' or 'result' (these change often)
	cards = soup.select('div.job_seen_beacon') # more modern indeed structure
	for c in cards:
		try:
			# job id
			job_id = c.get('data-jk') or c.get('id')
			# title
			title_tag = c.select_one('h2.jobTitle')
			title = title_tag.get_text(strip=True) if title_tag else None
			# company
			company_tag = c.select_one('span.companyName')
			company = company_tag.get_text(strip=True) if company_tag else None
			# location
			loc_tag = c.select_one('div.companyLocation')
			location = loc_tag.get_text(strip=True) if loc_tag else None
			# summary/snippet
			summary_tag = c.select_one('div.job-snippet')
			summary = summary_tag.get_text(separator=' ', strip=True) if summary_tag else None
			# post date
			post_tag = c.select_one('span.date')
			post_date = post_tag.get_text(strip=True) if post_tag else None
			# url
			a = c.select_one('a')
			href = a.get('href') if a else None
			url = urljoin(base_url, href) if href else None

			results.append({
				'job_id': job_id,
				'title': title,
				'company': company,
				'location': location,
				'summary': summary,
				'post_date': post_date,
				'url': url,
				'raw_html': str(c),
			})
		except Exception:
			logging.exception('Failed to parse card')
	return results

## 5) Page fetcher with pagination
- For Indeed, the query parameters include `q` (query), `l` (location), and `start` (offset). Adjust per site.

In [17]:
def build_search_url(base: str, query: str, location: str, start: int = 0, results_per_page: int = 10) -> str:
	params = {
		'q': query,
		'l': location,
		'start': start,
	}
	return f"{base}?{urlencode(params)}"

In [18]:
def fetch_page_requests(session: requests.Session, url: str, timeout: int = 15) -> Optional[str]:
	try:
		resp = session.get(url, timeout=timeout)
		resp.raise_for_status()
		return resp.text
	except requests.RequestException:
		logging.exception('Request failed for %s', url)
		return None

### Selenium fallback (if the site needs JS rendering)

In [19]:
def fetch_page_selenium(url: str, headless: bool = True, wait: int = 2) -> Optional[str]:
	if not SELENIUM_AVAILABLE:
		logging.error('Selenium not available')
		return None
	options = Options()
	if headless:
		options.add_argument('--headless=new')
	options.add_argument('--no-sandbox')
	options.add_argument('--disable-dev-shm-usage')
	options.add_argument('--disable-gpu')
	options.add_argument('--lang=en-US')
	driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
	try:
		driver.get(url)
		time.sleep(wait)
		html = driver.page_source
		return html
	except Exception:
		logging.exception('Selenium fetch failed')
		return None
	finally:
		driver.quit()

## 6) High-level runner: iterate pages, parse, save

In [20]:
def run_scraper(config: Dict, parser=parse_indeed_search_page):
	session = get_session()
	init_db(config['sqlite_db'])

	results_total = 0
	for page in tqdm(range(config['max_pages']), desc='Pages'):
		start = page * config['results_per_page']
		url = build_search_url(config['base_url'], config['query'], config['location'], start, config['results_per_page'])
		logging.info(f"Fetching page {page+1}: {url}")
		html = fetch_page_requests(session, url)
		if not html:
			logging.warning('Requests fetch failed, trying selenium fallback...')
			html = fetch_page_selenium(url)
		if not html:
			logging.error('Both requests and selenium failed for page %s', page+1)
			continue

		jobs = parser(html, base_url='https://www.indeed.com')
		inserted = save_jobs_to_db(jobs, db_path=config['sqlite_db'])
		results_total += len(jobs)

		# polite delay
		delay = random.uniform(config['delay_min'], config['delay_max'])
		logging.info(f"Sleeping {delay:.2f}s")
		time.sleep(delay)

	logging.info(f"Scraping done. Parsed {results_total} job cards (new inserted tracked separately).")

## 7) Example: run the scraper (dry run)

Uncomment the next cell to run. For testing, set `max_pages` small.

In [ ]:
# Example run (uncomment to execute)
# CONFIG['query'] = 'data analyst'
# CONFIG['location'] = 'Cairo'
# CONFIG['max_pages'] = 2
# run_scraper(CONFIG)

## 8) Inspect DB and export to CSV

In [21]:
def read_jobs_from_db(db_path: str = CONFIG['sqlite_db'], limit: int = 100):
	conn = sqlite3.connect(db_path)
	df = pd.read_sql_query('SELECT * FROM jobs ORDER BY scraped_at DESC LIMIT ?', conn, params=(limit,))
	conn.close()
	return df

## 9) Improvements & production notes
- Use rotating proxies if scraping at scale.
- Add retries with exponential backoff.
- Respect robots.txt and site's terms.
- Implement more robust deduplication using hashes of (title+company+location) or use site job IDs.
- Consider using Scrapy for large-scale crawls with built-in throttling and pipelines.
- For scheduling: use Airflow, cron, or serverless functions. Keep in mind rate limits and legality.

> End of notebook. Adapt the parser to the current site's HTML structure before running.